# Unit 5, Lecture 3: Observability, seeing inside a running agent

Testing and validation happen **before** you ship. But once the agent is live and
real users hit it, how do you see what it does? A user says "it was slow" and you
were not watching. **You cannot fix what you cannot see.**

Observability is building the agent to tell you what it does as it runs. Three
pillars:

1. A **trace**, the timed steps of one run, to find the slow step.
2. **Metrics**, numbers across many runs, to watch overall health.
3. Structured **logs**, searchable records, to find the one bad run.

**Metrics alert, logs locate, a trace explains.** Timing is injected, so the trace
is real but this notebook is deterministic and never flakes.

## 1. A trace: the timed steps of one run

Record each step and its duration, then find where the time went. In production
the durations come from a real clock; here they are passed in (the L1 seam idea,
applied to time).

In [ ]:
from cse476.observability import trace_run

# an agent that classifies, looks up a policy, then decides
trace = trace_run([
    ("classify",      10.0),
    ("lookup_policy", 30.0),   # the slow step
    ("decide",         5.0),
])

print("path:   ", trace.path())
print("total:  ", trace.total_ms(), "ms")
print("slowest:", trace.slowest().step, f"({trace.slowest().ms} ms)")
print()
print("-> optimise the policy lookup; it is 2/3 of the run. Anything else is waste.")

## 2. Metrics: numbers across many runs

A trace is one user; metrics are everyone. Fold each run into running totals to
get the headline latency and failure numbers, the ones you put on a dashboard.

In [ ]:
from cse476.observability import Metrics, trace_run

metrics = Metrics()
metrics.observe(trace_run([("a", 10.0), ("b", 10.0)]))                # 20ms ok
metrics.observe(trace_run([("a", 30.0), ("b", 10.0)]), failed=True)   # 40ms FAILED
metrics.observe(trace_run([("a", 20.0), ("b", 10.0)]))                # 30ms ok

print("runs:          ", metrics.runs)
print("average latency:", metrics.average_ms(), "ms")
print("failure rate:   ", metrics.failure_rate(), "(1 in 3)")
print()
print("-> a failure rate creeping up is invisible in one trace, but screams here.")

## 3. Structured logs: searchable records

Not a print statement, a dict with named fields you can filter: every failed run,
every run over a second, every run that took a given path.

In [ ]:
from cse476.observability import log_line

line = log_line("run-123", trace, outcome="failed")
for field, value in line.items():
    print(f"{field:14}: {value}")
print()
print("-> because it is DATA, you can search it: find all outcome=='failed' runs.")

## The three pillars on one incident

This is why you need all three. In order:

1. **Metrics alert**: the dashboard shows failure rate jumped 1% -> 15% at 2pm.
2. **Logs locate**: you search failed runs after 2pm and find 200 of them.
3. **A trace explains**: you open one and see the policy lookup timing out.
4. **You fix it**: the policy service was down. You knew in minutes, not days.

No single pillar solves it alone. Without any of them, you hear it from angry
users, days later.

In [ ]:
from cse476.observability import OBSERVABILITY_MAP, the_three_pillars

for concept, meaning in OBSERVABILITY_MAP.items():
    print(f"{concept:24} ->  {meaning}")
print()
for pillar, role in the_three_pillars().items():
    print(f"{pillar:10}: {role}")

## Your turn

**1. Trace your agent.** Add timing around each step of one capstone run and build
a trace. Find the slowest step. Was it where you expected?

**2. Aggregate metrics.** Run your agent ten times, folding each into a `Metrics`
object. Report average latency and failure rate.

**3. Walk an incident.** Imagine your failure rate doubles overnight. Write the
three steps: which metric alerts you, what you search in the logs, what a trace
shows. That is your incident plan.

In [ ]:
# your work here
